# Day I3 — 관절 운동학: 마커 좌표로 시상면 각도 곡선 만들고 읽기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I3.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I3.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 분절각 → 관절각 → %GC 곡선

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# BMC data/WinterTableA1.txt: 머리글 2줄, 106행 × 18열, 좌표 단위 cm(원전 기준), x = 진행 방향, y = 수직
raw = pd.read_csv("data/bmc/WinterTableA1.txt", sep=r"\s+", header=None, skiprows=2).to_numpy()
frame, t = raw[:, 0].astype(int), raw[:, 1]
names = ["rib", "hip", "knee", "fibula", "ankle", "heel", "mt5", "toe"]
M = {nm: raw[:, 2 + 2 * i: 4 + 2 * i] for i, nm in enumerate(names)}     # 마커별 (x, y)
print("frame 수 %d, 기록 %.3f s, 표본화 %.2f Hz" % (len(t), t[-1], (len(t) - 1) / t[-1]))

def seg_angle(distal, proximal):
    """원위 마커 → 근위 마커 벡터가 +x(앞쪽 수평)와 이루는 각(도), 반시계 +"""
    d = M[proximal] - M[distal]
    return np.degrees(np.unwrap(np.arctan2(d[:, 1], d[:, 0])))

thigh = seg_angle("knee", "hip")          # 대퇴: knee → hip
leg   = seg_angle("ankle", "fibula")      # 하퇴: ankle → fibula head  (toe·knee가 아님!)
foot  = seg_angle("mt5", "heel")          # 발:   5th metatarsal → heel
trunk = seg_angle("hip", "rib")           # 몸통(½HAT): hip → rib cage

knee  = thigh - leg                       # + = 굴곡, 0 = 완전 신전
ankle = foot - leg - 90                   # + = 배측굴곡(DF), − = 족저굴곡(PF)  ← 이 레슨의 규약
thigh_v = thigh - 90                      # 수직선 기준 대퇴각(Perry의 thigh 각): + = 굴곡
hip   = thigh - trunk                     # 몸통선 기준 대퇴각: + = 굴곡 (골반 기준 hip 각과는 다르다)

# [Wi] 부록 A가 표시한 사건: 오른발 heel contact = frame 28, 97 / toe-off = frame 1, 70
hc1, to, hc2 = 28, 70, 97
idx = np.arange(hc1 - 1, hc2)             # frame 28…97 (배열 위치는 frame − 1)
gc = (frame[idx] - hc1) / (hc2 - hc1) * 100
print("stride %.3f s, stance %.1f%%GC" % (t[hc2 - 1] - t[hc1 - 1], (to - hc1) / (hc2 - hc1) * 100))

def at(y, pct):                           # 특정 %GC에서의 값(선형 보간)
    return np.interp(pct, gc, y[idx])

print("\n%GC   knee(굴곡+)  ankle(DF+)  thigh_v(굴곡+)  hip_vs_trunk")
for p in [0, 12, 31, 45, 50, 62, 75, 87]:
    print("%3d   %8.1f   %9.1f   %10.1f   %10.1f" % (p, at(knee, p), at(ankle, p), at(thigh_v, p), at(hip, p)))
print("\n최댓값/최솟값 (한 주기)")
for nm, y in [("knee", knee), ("ankle", ankle), ("thigh_v", thigh_v)]:
    yy = y[idx]
    print("%-8s max %5.1f at %4.1f%%GC | min %6.1f at %4.1f%%GC"
          % (nm, yy.max(), gc[yy.argmax()], yy.min(), gc[yy.argmin()]))

fig, axs = plt.subplots(1, 3, figsize=(11, 3.2), sharex=True)
for ax, y, title in zip(axs, [knee, ankle, thigh_v],
                        ["knee (flexion +)", "ankle (dorsiflexion +)", "thigh vs vertical (flexion +)"]):
    ax.plot(gc, y[idx], "k")
    ax.axvline((to - hc1) / (hc2 - hc1) * 100, color="r", ls="--", lw=0.8)   # toe-off
    ax.axhline(0, color="gray", lw=0.5)
    ax.set_title(title); ax.set_xlabel("% gait cycle (right heel contact = 0)")
axs[0].set_ylabel("angle (deg)")
out = Path("figures/advanced/I/I3_joint_angles.png")
out.parent.mkdir(parents=True, exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=150)
print("저장:", out)

### 예시 2 — 마커를 바꾸면, 부호를 바꾸면

In [ ]:
import numpy as np
import pandas as pd

raw = pd.read_csv("data/bmc/WinterTableA1.txt", sep=r"\s+", header=None, skiprows=2).to_numpy()
names = ["rib", "hip", "knee", "fibula", "ankle", "heel", "mt5", "toe"]
M = {nm: raw[:, 2 + 2 * i: 4 + 2 * i] for i, nm in enumerate(names)}

def seg_angle(distal, proximal):
    d = M[proximal] - M[distal]
    return np.degrees(np.unwrap(np.arctan2(d[:, 1], d[:, 0])))

leg_fib  = seg_angle("ankle", "fibula")
leg_knee = seg_angle("ankle", "knee")
foot_mt5 = seg_angle("mt5", "heel")
foot_toe = seg_angle("toe", "heel")

ankle_ref = foot_mt5 - leg_fib - 90            # 이 레슨의 기준 정의(DF +)
ankle_alt = foot_toe - leg_knee - 90           # 마커를 바꾼 정의
diff = ankle_alt - ankle_ref
print("마커를 바꾸면 발목각이 평균 %+.1f° 이동 (범위 %+.1f ~ %+.1f°)" % (diff.mean(), diff.min(), diff.max()))

ankle_text = leg_fib - foot_mt5 + 90           # [Wi] 본문 3.6.2 식의 부호(PF +)
print("본문 식(PF +)과 이 레슨 정의(DF +)의 합 = %.1f (항상 0이면 부호만 반대)" % np.abs(ankle_text + ankle_ref).max())